# Ch 8. Attention 다시 보기

scaled dot-product attention을 손으로 짜고, PyTorch `F.scaled_dot_product_attention`과 결과를 비교한다.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part3/ch08_attention.ipynb)

In [ ]:
# 필요한 경우 설치
# !pip install torch

import torch
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'device: {device}')
print(f'torch version: {torch.__version__}')

## 최소 예제 — 손으로 짜기 30줄

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

5 단계:
1. **Q · Kᵀ** — 토큰 쌍의 dot product. shape `(seq, seq)`
2. **÷ √d_k** — 점수 분산 안정
3. **causal mask** — 미래 위치는 -∞
4. **softmax** — 확률 분포로
5. **× V** — value 가중 합

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, D = 1, 4, 8                          # batch, seq, hidden
x = torch.randn(B, T, D)

# 학습 가능한 투영 — 실모델은 nn.Linear, 여기선 노출용으로 직접
Wq = torch.randn(D, D); Wk = torch.randn(D, D); Wv = torch.randn(D, D)
Q = x @ Wq                                 # (B, T, D)
K = x @ Wk
V = x @ Wv

scores = Q @ K.transpose(-2, -1)           # (B, T, T)        (1)
scores = scores / (D ** 0.5)               #                  (2)

# Causal mask: 위치 i 는 j > i 를 못 보게
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()  # (T, T)  (3)
scores = scores.masked_fill(mask, float('-inf'))

attn = F.softmax(scores, dim=-1)           # (B, T, T)
out = attn @ V                             # (B, T, D)         (4)
print("attention weights row 0:", attn[0, 0])  # 위치 0 은 자기만 봄 → [1, 0, 0, 0]
print("attention weights row 1:", attn[0, 1])
print("attention weights row 2:", attn[0, 2])
print("attn shape:", attn.shape)
print("out shape:", out.shape)

## 실전 — `F.scaled_dot_product_attention` 한 줄과 비교

PyTorch 2.x 부터 `F.scaled_dot_product_attention` 한 줄에 같은 연산.
내부적으로 **FlashAttention** (Dao et al., 2022) 또는 효율 구현 자동 선택.

- `is_causal=True` 면 mask 자동 적용
- 같은 결과 — 1e-6 미만 차이는 부동소수점 오차

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, D = 1, 4, 8
x = torch.randn(B, T, D)
Wq = torch.randn(D, D); Wk = torch.randn(D, D); Wv = torch.randn(D, D)
Q, K, V = x @ Wq, x @ Wk, x @ Wv

# 한 줄                                                       (1)
out_fast = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

# 손으로 짠 것                                                 (2)
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
scores = (Q @ K.transpose(-2, -1)) / (D ** 0.5)
scores = scores.masked_fill(mask, float('-inf'))
out_manual = F.softmax(scores, dim=-1) @ V

print("max abs diff:", (out_fast - out_manual).abs().max().item())  # 1e-6 수준

## multi-head — 그저 reshape

`d_model=64, n_head=8` 이면 head 마다 `head_dim = 8`.
각 head 가 독립적으로 attention 한 다음 concat.

`view + transpose` 두 줄. 새 알고리즘이 아니라 **차원 쪼개기**.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, D, H = 1, 4, 64, 8
head_dim = D // H                           # 8

# 무작위 Q, K, V 생성 (B, T, D)
Q = torch.randn(B, T, D)
K = torch.randn(B, T, D)
V = torch.randn(B, T, D)

# (B, T, D) → (B, T, H, head_dim) → (B, H, T, head_dim)
def split(x):
    return x.view(B, T, H, head_dim).transpose(1, 2)

Q, K, V = split(Q), split(K), split(V)                          # (1)
out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)    # 자동 broadcast (2)
out = out.transpose(1, 2).contiguous().view(B, T, D)             # 다시 합치기
print("multi-head out shape:", out.shape)  # (1, 4, 64)

## 연습문제

1. §4 의 5 줄 attention 을 batch B=2, seq T=8, hidden D=16 으로 돌려보고 `attn` shape 와 합 (`attn.sum(-1)`) 이 모두 1 인지 확인하라.
2. §5 의 SDPA 한 줄 결과와 수동 결과의 차이를 다양한 dtype (fp32, fp16, bf16) 으로 비교. 어느 dtype 이 가장 차이가 큰가?
3. causal mask 를 **반대로** (`triu(diagonal=0)` — 자기 포함 미래만 봄) 적용하면 모델이 어떻게 학습될까? 한 epoch 돌려 손실 곡선을 정상 mask 와 비교.
4. multi-head 8개를 모두 같은 weight 로 초기화하면 무엇이 일어날까? PyTorch 기본 초기화가 head 마다 다른 결과를 자동 보장하는 이유는?
5. **(생각해볼 것)** seq=10K 인 모델에서 attention `O(n²)` 가 메모리 100MB 를 잡는다. seq=100K 면 단순 계산으로 10GB. FlashAttention 은 어떻게 이 문제를 해결하는가? (한 문단으로 핵심만)

In [ ]:
# 연습문제 1: B=2, T=8, D=16 으로 attention 실행 & attn.sum(-1) 확인


In [ ]:
# 연습문제 2: dtype 별 (fp32, fp16, bf16) SDPA vs 수동 차이 비교


In [ ]:
# 연습문제 3: triu(diagonal=0) 로 causal mask 반대 적용 실험


In [ ]:
# 연습문제 4: 같은 weight 로 초기화한 multi-head 동작 확인


In [ ]:
# 연습문제 5: FlashAttention 메모리 절감 원리 — 메모나 코드로 정리
